# Conversão de anotações VOC → YOLO

Valida que `voc_to_yolo.py` converte as anotações Pascal VOC (bbox absoluto) para o formato YOLO (normalizado) sem perder geometria, usando uma amostra real do dataset.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path("../../..").resolve()
sys.path.insert(0, str(REPO_ROOT / "src" / "preprocessing"))

from voc_to_yolo import parse_annotation, to_yolo_line, DENVER_CLASSES

RAW_ROOT = REPO_ROOT / "data" / "raw" / "24_chromosomes_object"
print(f"{len(DENVER_CLASSES)} classes reais (grupo de Denver): {DENVER_CLASSES}")


24 classes reais (grupo de Denver): ['A1', 'A2', 'A3', 'B4', 'B5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'D13', 'D14', 'D15', 'E16', 'E17', 'E18', 'F19', 'F20', 'G21', 'G22', 'X', 'Y']


Pega uma anotação de amostra e converte cada bbox para uma linha YOLO (classe única, compatível com `nc: 1`).

In [2]:
xml_sample = next((RAW_ROOT / "annotations").glob("*.xml"))
image_sample = RAW_ROOT / "JEPG" / f"{xml_sample.stem}.jpg"

annotation = parse_annotation(xml_sample)
print(f"Amostra: {xml_sample.name} ({annotation['width']}x{annotation['height']}, {len(annotation['objects'])} objetos)")

yolo_lines = [to_yolo_line(obj, annotation["width"], annotation["height"]) for obj in annotation["objects"]]
for line in yolo_lines[:5]:
    print(line)


Amostra: 1057924.xml (524x869, 46 objetos)
0 0.217557 0.860759 0.248092 0.156502
0 0.612595 0.795742 0.347328 0.093211
0 0.811069 0.433257 0.091603 0.205984
0 0.361641 0.399885 0.227099 0.201381
0 0.827290 0.207710 0.192748 0.079402


Desenha os bboxes reconstruídos a partir do YOLO normalizado sobre a imagem original, pra conferir visualmente que a conversão preserva a geometria.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.image as mpimg

img = mpimg.imread(image_sample)
fig, ax = plt.subplots(1, figsize=(8, 12))
ax.imshow(img)

for obj, line in zip(annotation["objects"], yolo_lines, strict=True):
    _, xc, yc, w, h = (float(v) if i > 0 else v for i, v in enumerate(line.split()))
    xc, yc, w, h = xc * annotation["width"], yc * annotation["height"], w * annotation["width"], h * annotation["height"]
    rect = patches.Rectangle(
        (xc - w / 2, yc - h / 2), w, h, linewidth=1, edgecolor="lime", facecolor="none"
    )
    ax.add_patch(rect)
    ax.text(xc - w / 2, yc - h / 2 - 2, obj["name"], color="lime", fontsize=6)

ax.set_title(f"{xml_sample.stem}: bboxes reconstruídos a partir do YOLO normalizado")
ax.axis("off")
plt.show()
